In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

try:
    from scipy.interpolate import PchipInterpolator
except Exception:  # pragma: no cover
    PchipInterpolator = None

from .model import PCSSOR


def build_anchor_geometry(anchor_source_df: pd.DataFrame) -> pd.DataFrame:
    """Build trusted per-surface anchor grid from training geometry."""
    parts = []
    for side in ["upper", "lower"]:
        s = anchor_source_df[anchor_source_df["side"] == side].copy()
        s = s.groupby(["x", "side"], as_index=False)["y"].mean().sort_values("x")
        parts.append(s[["x", "y", "side"]])
    return pd.concat(parts, ignore_index=True)


def predict_anchor_curve(model: PCSSOR, anchor_geom_df: pd.DataFrame, alpha_value: float) -> pd.DataFrame:
    rows = []
    for side in ["upper", "lower"]:
        s = anchor_geom_df[anchor_geom_df["side"] == side].sort_values("x").copy()
        if len(s) == 0:
            continue
        s["Alpha"] = float(alpha_value)
        rows.append(s[["x", "y", "Alpha", "side"]])
    out = pd.concat(rows, ignore_index=True)
    out["Cp_anchor_pred"] = model.predict(out)
    return out


def _make_interpolator(x_anchor, y_anchor):
    x_anchor = np.asarray(x_anchor, dtype=float)
    y_anchor = np.asarray(y_anchor, dtype=float)
    order = np.argsort(x_anchor)
    x_anchor = x_anchor[order]
    y_anchor = y_anchor[order]
    x_unique, idx = np.unique(x_anchor, return_index=True)
    y_unique = y_anchor[idx]
    if len(x_unique) < 2:
        return None, "failed"
    if PchipInterpolator is not None and len(x_unique) >= 3:
        return PchipInterpolator(x_unique, y_unique, extrapolate=False), "anchor_grid_pchip"

    def interp_fn(x_new):
        return np.interp(np.asarray(x_new, dtype=float), x_unique, y_unique, left=np.nan, right=np.nan)

    return interp_fn, "anchor_grid_linear"


def dense_anchor_pchip_inference(
    model: PCSSOR,
    future_df: pd.DataFrame,
    anchor_source_df: pd.DataFrame,
    q_90: float | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Predict on training-consistent anchor grid and interpolate to dense/future grid."""
    future_df = future_df.copy()
    anchor_geom = build_anchor_geometry(anchor_source_df)
    future_df["Cp_pred_direct_raw"] = model.predict(future_df)
    final_parts = []
    anchor_parts = []

    for alpha in sorted(future_df["Alpha"].unique()):
        future_a = future_df[future_df["Alpha"] == alpha].copy()
        anchor_pred = predict_anchor_curve(model, anchor_geom, float(alpha))
        anchor_parts.append(anchor_pred.copy())

        for side in ["upper", "lower"]:
            f_side = future_a[future_a["side"] == side].sort_values("x").copy()
            a_side = anchor_pred[anchor_pred["side"] == side].sort_values("x").copy()
            if len(f_side) == 0 or len(a_side) < 2:
                continue
            interp_fn, method = _make_interpolator(a_side["x"].values, a_side["Cp_anchor_pred"].values)
            if interp_fn is None:
                f_side["Cp_pred"] = np.nan
                f_side["pred_method"] = method
            else:
                xq = f_side["x"].values.astype(float)
                yq = interp_fn(xq)
                x_min, x_max = float(a_side["x"].min()), float(a_side["x"].max())
                y_min = float(a_side.loc[a_side["x"].idxmin(), "Cp_anchor_pred"])
                y_max = float(a_side.loc[a_side["x"].idxmax(), "Cp_anchor_pred"])
                yq = np.where(xq < x_min, y_min, yq)
                yq = np.where(xq > x_max, y_max, yq)
                f_side["Cp_pred"] = yq
                f_side["pred_method"] = method
            final_parts.append(f_side)

    final = pd.concat(final_parts, ignore_index=True)
    anchors = pd.concat(anchor_parts, ignore_index=True)
    final["alpha_in_train_range"] = (final["Alpha"] >= model.model_["alpha_min"]) & (final["Alpha"] <= model.model_["alpha_max"])
    final["alpha_warning"] = np.where(final["alpha_in_train_range"], "inside_train_range", "outside_train_range_clipped")
    if q_90 is not None:
        final["Cp_lo_90"] = final["Cp_pred"] - q_90
        final["Cp_hi_90"] = final["Cp_pred"] + q_90
    return final, anchors
